# Clase 02 — Texto bruto y tokenización

**Curso:** NLP Clásico con Python · **Bloque 1:** Del texto a los tokens
**Tiempo estimado:** 105 minutos · **Capítulo del manual:** `manual/02_texto_crudo_tokenizacion.md`

## Qué vas a ver en esta clase

- **Unicode**: por qué un mismo carácter acentuado puede tener dos representaciones (NFC y NFD), cómo lo delata `len()` y cuándo quitar acentos es un error.
- **Limpieza con expresiones regulares** (`re`) sin destruir información útil: URLs, menciones, alargamientos, emojis, cifras, restos de marcado. Construirás una función `limpiar(texto)` paso a paso.
- **Tokenización**: por qué `split()` no basta; `re.findall`, NLTK y spaCy frente a frente; segmentación en frases.
- **Tokens y tipos**: tamaño del vocabulario, frecuencias, hapax y la distribución rango-frecuencia en dos textos literarios de dominio público y en las reseñas.
- **Medir la calidad de una limpieza** contra la columna `texto_limpio` (referencia) de las reseñas.
- **Otras dificultades del lenguaje** (anáfora, pragmática, variación) y por qué ninguna limpieza las resuelve.

## Requisitos

- Python básico y la clase 1 (la pipeline clásica y el paso 3 «ingenuo» que aquí se arregla).
- Librerías: `pandas`, `matplotlib`, `nltk` (con los recursos `punkt`) y `spaCy` (solo su tokenizador, sin modelos). La celda siguiente las instala solo si faltan (funciona igual en Google Colab).
- La carpeta `datos/` del curso (en Colab, súbela al panel de archivos).

> **Cómo usar este notebook:** ejecuta las celdas en orden. Todas las cifras son de estos corpus pequeños y de estilo controlado (*ejemplo ilustrativo*): no se generalizan a textos reales de cualquier origen.

---
## Instalación y recursos

Esta celda instala lo que falte y descarga los recursos de NLTK. Con todo instalado no hace nada.

In [ ]:
import subprocess, sys

def asegurar(paquete, modulo):
    """Instala el paquete solo si falta (idempotente y silencioso)."""
    try:
        __import__(modulo)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquete], check=True)

for paquete, modulo in (("pandas", "pandas"), ("matplotlib", "matplotlib"), ("nltk", "nltk"), ("spacy", "spacy")):
    asegurar(paquete, modulo)

import nltk

for recurso in ("punkt", "punkt_tab"):
    try:
        nltk.data.find(f"tokenizers/{recurso}")
    except LookupError:
        nltk.download(recurso, quiet=True)

print("Dependencias y recursos de NLTK listos (spaCy solo se usa su tokenizador, sin descargar modelos).")

---
## Celda 0 — Setup

Importamos las librerías, fijamos la semilla y definimos `cargar_datos(nombre)`, que busca el archivo en `datos/`, `../datos/` y `/content/datos/`. Cargamos las 300 reseñas de cada idioma y los dos extractos literarios de dominio público (el capítulo I del *Quijote* y las primeras ≈ 1 900 palabras del capítulo I de *I promessi sposi*). Son textos de siglos y estilos distintos: sirven para ver el comportamiento del texto «limpio» y del ruidoso.

In [ ]:
import os
import re
import unicodedata
import difflib
import random
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import spacy
from nltk.tokenize import word_tokenize, sent_tokenize

random.seed(42)                          # semilla fija: todos obtenemos los mismos resultados
np.random.seed(42)
plt.rcParams["figure.figsize"] = (9, 4)
plt.rcParams["axes.grid"] = True
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 90)

def cargar_datos(nombre):
    """Busca un archivo en datos/, ../datos/ y /content/datos/ y devuelve su ruta."""
    for base in ("datos", "../datos", "/content/datos"):
        ruta = os.path.join(base, nombre)
        if os.path.exists(ruta):
            return ruta
    # Copia pubblica (Colab): se non c'è in locale, scarica il file dal repository dei notebook.
    try:
        import urllib.request
        destino = os.path.join("datos", nombre)
        os.makedirs(os.path.dirname(destino), exist_ok=True)
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/RaffaeleBini/arcadia-notebooks/main/14-nlp-classico/datos/{nombre}", destino)
        return destino
    except Exception:
        pass
    raise FileNotFoundError(f"No encuentro '{nombre}'. Sube la carpeta datos/ al entorno (en Colab: panel de archivos) o déjala junto al curso.")

resenas = {"spa": pd.read_csv(cargar_datos("resenas_es.csv")), "ita": pd.read_csv(cargar_datos("resenas_it.csv"))}
libros = {}
for idioma, archivo in (("spa", "gutenberg/quijote_cap1.txt"), ("ita", "gutenberg/promessi_sposi_cap1.txt")):
    with open(cargar_datos(archivo), encoding="utf-8") as f:
        libros[idioma] = f.read()

print({i: len(d) for i, d in resenas.items()}, {i: len(t) for i, t in libros.items()}, "(reseñas y caracteres de cada libro)")

---
## 1. Unicode: qué es realmente un «carácter»

### 1.1 Puntos de código y bytes

Unicode da a cada símbolo un **punto de código** (`U+00F3` para `ó`). UTF-8 es la forma de guardarlos en bytes: las letras ASCII ocupan 1 byte y las acentuadas 2 (los emojis, 4). Por eso `len()` cuenta puntos de código y **no** bytes ni «lo que ve el ojo».

In [ ]:
# datos-es
PALABRAS_UNICODE = {"spa": "canción", "ita": "città"}

In [ ]:
for idioma, palabra in PALABRAS_UNICODE.items():
    print(idioma, palabra, "→", len(palabra), "puntos de código,", len(palabra.encode("utf-8")), "bytes en UTF-8")

for c in PALABRAS_UNICODE["ita"]:
    print(f"{c!r:5} U+{ord(c):04X}  {unicodedata.name(c)}")

emoji = "\U0001F60A"
print(emoji, len(emoji), "punto de código,", len(emoji.encode("utf-8")), "bytes")

### 1.2 Experimento guiado: dos formas de escribir la misma palabra

`é` puede guardarse como **un** punto de código (forma compuesta, **NFC**: `U+00E9`) o como **dos**: la `e` y un acento combinante `U+0301` (forma descompuesta, **NFD**). Las dos se dibujan igual. Comparamos las dos versiones de `canción`.

In [ ]:
nfc = unicodedata.normalize("NFC", PALABRAS_UNICODE["spa"])
nfd = unicodedata.normalize("NFD", PALABRAS_UNICODE["spa"])

print(nfc, nfd, "→ se ven igual, pero len:", len(nfc), "vs", len(nfd))
print("¿Son iguales para Python?", nfc == nfd)
print("Puntos de código de la versión NFD:", [f"U+{ord(c):04X}" for c in nfd])

indice = {nfc: "clave guardada en NFC"}
print("¿Encuentro la versión NFD en el diccionario?", nfd in indice)
print("¿Y si normalizo antes de buscar?", unicodedata.normalize("NFC", nfd) in indice)

**¿Qué observas?** Las dos cadenas se ven idénticas, pero tienen longitudes distintas (7 y 8) y `==` da `False`. Un diccionario, un `Counter` o un vocabulario tratarían `canción` como **dos palabras**. Esto ocurre en la práctica: los nombres de archivo de macOS y algunos textos copiados de PDF o de la web llegan en NFD. **Regla:** normaliza todo a NFC nada más leerlo.

Existe además **NFKC**, que aplica también equivalencias de «compatibilidad» (ligaduras, exponentes, fracciones). Es útil, pero cambia más cosas (`²` → `2`), así que se usa con criterio.

In [ ]:
# datos-es
COMPATIBILIDAD = ["ﬁnestra", "m²", "½", "…"]

In [ ]:
for texto in COMPATIBILIDAD:
    print(f"{texto!r:12} NFC → {unicodedata.normalize('NFC', texto)!r:12} NFKC → {unicodedata.normalize('NFKC', texto)!r}")

### 1.3 `lower()` frente a `casefold()`

Para **comparar** textos sin distinguir mayúsculas, `casefold()` es más agresivo que `lower()`: pensado para comparar, no para mostrar. En español e italiano casi siempre coinciden, pero hay casos reales (ligaduras copiadas de PDF, la `ß` alemana en nombres propios).

In [ ]:
# datos-es
CASOS_MAYUS = [("Straße", "STRASSE"), ("ﬁnestra", "FINESTRA"), ("Roma", "ROMA")]

In [ ]:
for a, b in CASOS_MAYUS:
    print(f"{a!r:10} {b!r:10}  lower ==: {a.lower() == b.lower()!s:5}  casefold ==: {a.casefold() == b.casefold()}")

**¿Qué observas?** Con `lower()` la ligadura `ﬁ` y la `ß` no coinciden con sus equivalentes; con `casefold()` sí. Pero cuidado con la otra cara: al pasar todo a minúsculas `Rosa` (nombre) y `rosa` (color) son la **misma** cadena, y `Marino` (apellido) y `marino` (adjetivo) también. Pasar a minúsculas es una decisión, no un automatismo.

### 1.4 Quitar acentos… ¿y cuándo no?

Quitar acentos reduce variantes (`rapido`/`rápido`), pero **borra distinciones**: `papá`/`papa`, `sí`/`si`, `él`/`el`, `è`/`e`, `dà`/`da`. Y una implementación ingenua se carga la `ñ`: `año` → `ano`. Escribimos una versión que protege la `ñ` y medimos cuántas palabras **distintas** del vocabulario de las reseñas limpias se funden en una sola al quitar los acentos.

In [ ]:
def quitar_acentos(texto):
    """Quita los acentos (marcas combinantes) y conserva la ñ."""
    salida = []
    for c in unicodedata.normalize("NFD", texto):
        if unicodedata.category(c) == "Mn" and not (c == "̃" and salida and salida[-1] in "nN"):
            continue
        salida.append(c)
    return unicodedata.normalize("NFC", "".join(salida))

def tipos_en_colision(textos):
    """Agrupa las palabras distintas que coinciden al quitar los acentos."""
    frecuencia = Counter(p for t in textos for p in re.findall(r"\w+", t.lower()))
    grupos = {}
    for palabra in frecuencia:
        grupos.setdefault(quitar_acentos(palabra), []).append(palabra)
    return {k: sorted(v, key=lambda p: -frecuencia[p]) for k, v in grupos.items() if len(v) > 1}, len(frecuencia)

print(quitar_acentos("canción, año, pequeño, città, perché"))   # dato-es
for idioma in ("spa", "ita"):
    colisiones, n_tipos = tipos_en_colision(resenas[idioma]["texto_limpio"])
    ejemplos = sorted(colisiones.values(), key=lambda v: -len(v))[:6]
    print(idioma, f"→ {len(colisiones)} grupos de palabras que se funden, sobre {n_tipos} tipos:", ejemplos)

**¿Qué observas?** Incluso en un corpus pequeño hay grupos de palabras distintas que quedan iguales sin acentos (`que`/`qué`, `se`/`sé`, `llevo`/`llevó` en español; `e`/`è`, `ne`/`né` en italiano). En la clase 6 (sentimiento) una negación o un adverbio de una sola letra puede cambiar la polaridad. **Regla práctica:** quita acentos solo si tus textos vienen con acentos inconsistentes (como los de las reseñas ruidosas) y tu tarea lo tolera; en tareas de análisis gramatical (clase 7) no lo hagas.

### 1.5 Apóstrofos, comillas tipográficas y caracteres invisibles

Copiar y pegar desde Word, PDF o la web trae comillas rizadas (`’ “ ”`), guiones largos y caracteres invisibles (espacio de ancho cero `U+200B`, guion blando `U+00AD`). Para el ojo son inocuos; para un `split()` o una comparación, no: `l’uomo` con apóstrofo curvo y `l'uomo` con apóstrofo recto son cadenas distintas. Los normalizamos con `str.translate`.

In [ ]:
# datos-es
TEXTO_SUCIO = ["Dijo: “hola” y se fue…", "l’uomo dell’amico", "ca­sa​ y tra​bajo"]

In [ ]:
INVISIBLES = dict.fromkeys(map(ord, "​‌‍⁠﻿­"), None)
TIPOGRAFICAS = {ord("‘"): "'", ord("’"): "'", ord("´"): "'", ord("`"): "'",
                ord("“"): '"', ord("”"): '"'}

def normalizar_unicode(texto, conservar=False):
    """Paso 1: NFC, comillas y apóstrofos rectos, sin caracteres invisibles."""
    texto = unicodedata.normalize("NFC", texto)
    return texto.translate(INVISIBLES).translate(TIPOGRAFICAS)

for t in TEXTO_SUCIO:
    print(repr(t), len(t), "→", repr(normalizar_unicode(t)), len(normalizar_unicode(t)))

**¿Qué observas?** Las longitudes cambian y las cadenas quedan como las que escribirías a mano. Este será el primer paso de `limpiar`. Las «…» tipográficas quedan como están: no todo lo raro hay que cambiarlo.

---
## 2. Limpieza con expresiones regulares

**Principio guía: limpiar no es borrar.** Cada cosa que quitas la pierdes para siempre en el resto de la pipeline. Antes de eliminar MAYÚSCULAS, `!!!`, emojis o negaciones pregúntate qué información llevan (en la clase 6 verás que el sentimiento depende de ellas). Dos estrategias: **extraer** esa señal antes de limpiar (un contador de mayúsculas, de exclamaciones, de emojis) o **sustituirla** por un token (un emoji → `<emo_pos>`).

Construimos `limpiar(texto)` como una lista de **pasos con nombre**, cada uno una función pequeña y comprobable. El primero (`normalizar_unicode`) ya lo tienes. Empezamos con URLs y menciones.

In [ ]:
RE_URL = re.compile(r"(?:https?://|www\.)\S+")
RE_MENCION = re.compile(r"@\w+")
RE_HASHTAG = re.compile(r"#(\w+)")

def quitar_urls(texto, conservar=False):
    """Paso 2: elimina las direcciones web."""
    return RE_URL.sub(" ", texto)

def quitar_menciones_hashtags(texto, conservar=False):
    """Paso 3: quita @menciones; del #hashtag quita la marca (conservar=True deja la palabra)."""
    texto = RE_MENCION.sub(" ", texto)
    return RE_HASHTAG.sub(r"\1" if conservar else " ", texto)

MUESTRAS_WEB = ["@atencion_cliente ¡Gracias! Más info en https://t.co/aB12cD #recomendado", "Vedi www.esempio.test/r/4821 #consigliato, @supporto_utenti"]   # dato-es
for t in MUESTRAS_WEB:
    print(repr(quitar_menciones_hashtags(quitar_urls(t))), "|", repr(quitar_menciones_hashtags(quitar_urls(t), conservar=True)))

Ahora emojis y restos de marcado. En el extracto de *I promessi sposi* las palabras en cursiva aparecen entre guiones bajos (`_Resegone_`): un residuo de la edición electrónica que un tokenizador leería como parte de la palabra. Un emoji, en cambio, es información: lo sustituimos por un token según su polaridad *si `conservar=True`* (criterio propio y discutible: `🙄` es más bien irónico).

> **Atención:** los guiones bajos también aparecen **dentro** de nombres de usuario (`@atencion_cliente`). Por eso el patrón exige que el guion bajo de apertura y el de cierre no estén pegados a una letra por fuera (`(?<!\w)` y `(?!\w)`). La celda muestra, en la tercera columna, qué pasaría con la versión ingenua `_(.+?)_`.

In [ ]:
RE_EMOJI = re.compile("[\U0001F300-\U0001FAFF☀-➿⭐️]+")
EMOJI_POS = set("😊🙂👍😍👌🙌")
EMOJI_NEG = set("👎😡😤🤦🙄")
RE_MARKUP = re.compile(r"(?<!\w)_(\S(?:.*?\S)?)_(?!\w)")

def tratar_emojis(texto, conservar=False):
    """Paso 4: quita los emojis o los sustituye por <emo_pos>, <emo_neg> o <emo_neu>."""
    def cambiar(m):
        if not conservar:
            return " "
        g = m.group(0)
        clase = "pos" if any(c in EMOJI_POS for c in g) else "neg" if any(c in EMOJI_NEG for c in g) else "neu"
        return f" <emo_{clase}> "
    return RE_EMOJI.sub(cambiar, texto)

def quitar_markup(texto, conservar=False):
    """Paso 5: quita los _guiones bajos_ de cursiva que rodean un texto."""
    return RE_MARKUP.sub(r"\1", texto)

MUESTRAS_EMOJI = ["Me encantó 😍👍 pero el envío... 🙄", "Il _Resegone_, così chiamato, e _non lo fanno_.... tutto", "@soporte_ayuda no funciona_"]   # dato-es
for t in MUESTRAS_EMOJI:
    print(repr(tratar_emojis(t, conservar=True)), "|", repr(quitar_markup(t)), "|", repr(re.sub(r"_(.+?)_", r"\1", t)))

Cuatro pasos más, con sus trampas:

- **Alargamientos** (`buenísimooo`, `bellissimooo`): una letra repetida **tres o más veces** se reduce a una (en español e italiano ninguna palabra tiene tres letras iguales seguidas, pero sí *dos*: `llegó`, `bello`). Restringimos la regla a **letras**: con `\w` un `1000` se convertiría en `10`.
- **Puntuación repetida** (`!!!`, `???`): se reduce a un signo. La intensidad se pierde, por eso se extrae antes.
- **Espacios**: colapsar los múltiples y pegar la puntuación a la palabra anterior (`idea , ejecución` → `idea, ejecución`).

In [ ]:
RE_ALARGADA = re.compile(r"([^\W\d_])\1{2,}")

def reducir_alargamientos(texto, conservar=False):
    """Paso 6: reduce a una las letras repetidas tres o más veces (solo letras, nunca cifras)."""
    return RE_ALARGADA.sub(r"\1", texto)

def reducir_puntuacion(texto, conservar=False):
    """Paso 7: reduce !!! y ??? a un solo signo y los puntos suspensivos largos a tres."""
    texto = re.sub(r"([!?¡¿])\1+", r"\1", texto)      # dato-es
    return re.sub(r"\.{4,}", "...", texto)

def arreglar_espacios(texto, conservar=False):
    """Paso 8: colapsa espacios y pega la puntuación a la palabra anterior."""
    texto = re.sub(r"\s+", " ", texto)
    texto = re.sub(r"\s+([,;:.!?])", r"\1", texto)
    return texto.strip()

MUESTRAS_RUIDO = ["buenísimooo!!!  llegó tardeeee ...., pero 1000 gracias", "bellissimooo!!  davvero ???", "Buena idea , ejecución mejorable"]   # dato-es
for t in MUESTRAS_RUIDO:
    print(repr(arreglar_espacios(reducir_puntuacion(reducir_alargamientos(t)))))
print(repr(re.sub(r"(\w)\1{2,}", r"\1", "cuesta 1000 euros")), "← la versión con \\w rompe las cifras")

### 2.1 Ensamblar `limpiar` y ver el efecto de cada paso

Reunimos los ocho pasos en una lista con nombre. `mostrar_pasos` imprime el texto tras cada paso que lo cambia.

In [ ]:
PASOS_LIMPIEZA = [
    ("unicode", normalizar_unicode),
    ("URLs", quitar_urls),
    ("menciones y hashtags", quitar_menciones_hashtags),
    ("emojis", tratar_emojis),
    ("markup", quitar_markup),
    ("alargamientos", reducir_alargamientos),
    ("puntuación repetida", reducir_puntuacion),
    ("espacios", arreglar_espacios),
]

def limpiar(texto, conservar=False):
    """Limpia un texto aplicando los pasos en orden. conservar=True deja hashtags y emojis como señal."""
    for nombre, paso in PASOS_LIMPIEZA:
        texto = paso(texto, conservar)
    return texto

def mostrar_pasos(texto, conservar=False):
    print("original :", repr(texto))
    for nombre, paso in PASOS_LIMPIEZA:
        nuevo = paso(texto, conservar)
        if nuevo != texto:
            print(f"{nombre:>22} :", repr(nuevo))
        texto = nuevo

In [ ]:
# datos-es
RUIDOSAS = {"spa": "@atencion_cliente ¡¡BUENÍSIMOOOO!!!   El pedido llegó en 2 días por 4,99 € 😍😍 https://t.co/aB12cD #recomendado",
            "ita": "@assistenza Spedizione velocissimaaa!!!  Pagato 4,99 € e arrivato in 2 giorni 😍 https://t.co/aB12cD #consigliato"}

In [ ]:
mostrar_pasos(RUIDOSAS["spa"])
print()
mostrar_pasos(RUIDOSAS["ita"], conservar=True)

**¿Qué observas?** Cada paso hace una cosa y se puede probar por separado; si una regex es demasiado avara, el resultado intermedio delata **cuál**. Fíjate en que `4,99` sobrevive intacto (no hemos tocado las cifras) y en que con `conservar=True` el emoji se convierte en `<emo_pos>` y el hashtag deja su palabra.

Una trampa clásica: quitar «toda la puntuación» con `re.sub(r"[^\w\s]", "", texto)` convertiría `4,99` en `499`. Compruébalo abajo, y mira además qué señales expresivas se pierden al limpiar y cómo extraerlas **antes**.

In [ ]:
NEGADORES = {"spa": {"no", "nunca", "ni", "tampoco", "sin"}, "ita": {"non", "mai", "né", "senza", "nessun", "nessuno"}}   # dato-es

def senales(texto, idioma):
    """Cuenta señales expresivas ANTES de limpiar."""
    palabras = re.findall(r"\w+", texto)
    return {"mayúsculas": sum(1 for p in palabras if len(p) >= 3 and p.isupper()),
            "exclamaciones": texto.count("!"),
            "emojis": len(RE_EMOJI.findall(texto)),
            "alargadas": len(RE_ALARGADA.findall(texto)),
            "negaciones": sum(1 for p in palabras if p.lower() in NEGADORES[idioma])}

print(re.sub(r"[^\w\s]", "", "cuesta 4,99 euros"), "← el precio se ha roto")   # dato-es
for idioma in ("spa", "ita"):
    print(idioma, senales(RUIDOSAS[idioma], idioma))
    print("   tras limpiar:", senales(limpiar(RUIDOSAS[idioma]), idioma))

**¿Qué observas?** Tras `limpiar`, los emojis y las palabras alargadas caen a cero y las exclamaciones pasan de 3 a 1: **esa información ya no está en el texto**. Las MAYÚSCULAS siguen ahí porque `limpiar` no pasa a minúsculas; se pierden cuando lo hacemos (como en `palabras`). Si la tarea lo necesita (sentimiento), se calcula antes y se añade como rasgo aparte, o se conserva con un token. Las negaciones, en cambio, **no se tocan** en la limpieza: se decide qué hacer con ellas en las clases 3 (stopwords) y 6.

---
## 3. Tokenización

Un **token** es la unidad con la que trabaja el resto de la pipeline. Aquí tokenizamos en **palabras** (más la puntuación). ¿Basta `split()`? Probamos con frases que contienen las trampas del español y del italiano: puntuación pegada, apóstrofos con elisión (`l'uomo`, `un'ora`, `dell'amico`), contracciones (`del`, `al`), signos de apertura `¿ ¡`, cifras con coma y abreviaturas.

In [ ]:
# datos-es
FRASES_TOKEN = {
    "spa": "¿Cuánto cuesta el pan del Sr. García? ¡Solo 4,99 euros al día!",
    "ita": "L'uomo dell'amico ha aspettato un'ora, all'una e mezza: costava 4,99 euro.",
}

In [ ]:
for idioma, frase in FRASES_TOKEN.items():
    print(idioma, "split():", frase.split())

**¿Qué observas?** `García?` y `día!` llevan el signo pegado; `¿Cuánto` y `¡Solo` arrastran el de apertura; `un'ora,` junta artículo, nombre y coma. Una bolsa de palabras vería `día!` y `día` como palabras distintas.

### 3.1 Un tokenizador con `re.findall`

Un patrón con varias alternativas, de la más específica a la más general: (1) tokens con forma `<...>` (los que creamos al limpiar), (2) **números** con separadores (`4,99`, `1.250,50`), (3) **elisión**: palabra + apóstrofo delante de otra letra (`l'`, `dell'`, `un'`), (4) palabras (con guiones internos), (5) cualquier otro símbolo suelto (puntuación, `¿`, `¡`).

In [ ]:
PATRON_TOKEN = re.compile(r"<\w+>|\d+(?:[.,]\d+)*|\w+'(?=\w)|\w+(?:-\w+)*|[^\w\s]")

def tokenizar(texto):
    """Tokeniza en palabras, números, elisiones y signos de puntuación."""
    return PATRON_TOKEN.findall(texto)

for idioma, frase in FRASES_TOKEN.items():
    print(idioma, tokenizar(frase))

### 3.2 NLTK y spaCy frente a frente

Comparamos nuestro tokenizador con `nltk.tokenize.word_tokenize(language=...)` y con el tokenizador de spaCy (`spacy.blank`, sin modelo: solo reglas). **No hay una tokenización «correcta» universal**: cada herramienta toma decisiones distintas y hay que conocerlas.

In [ ]:
codigo_spacy = {"spa": "es", "ita": "it"}   # dato-es
nlp_blank = {idioma: spacy.blank(codigo) for idioma, codigo in codigo_spacy.items()}
nombre_nltk = {"spa": "spanish", "ita": "italian"}

def comparar_tokenizadores(texto, idioma):
    return {"split": texto.split(), "re.findall": tokenizar(texto),
            "nltk": word_tokenize(texto, language=nombre_nltk[idioma]),
            "spaCy": [t.text for t in nlp_blank[idioma](texto)]}

for idioma, frase in FRASES_TOKEN.items():
    print(idioma, frase)
    for herramienta, tokens in comparar_tokenizadores(frase, idioma).items():
        print(f"   {herramienta:>10} ({len(tokens):2}):", tokens)

**¿Qué observas?** (1) NLTK deja `¿Cuánto` y `¡Solo` pegados y **no separa** los apóstrofos del italiano (`l'uomo` queda entero). (2) spaCy separa `¿` y `¡` y corta `L'` / `uomo`, como hace nuestro patrón. (3) Ninguno parte `del` ni `al`: en español los tratamos como palabras (más adelante, con `es_core_news_sm`, verás que spaCy sí las descompone en `de + el`, un paso de análisis y no de tokenización pura). (4) Las cifras `4,99` se conservan enteras. (5) `Sr.` es un caso límite: con nuestro patrón queda `Sr` + `.`. Cifras de spaCy: con spaCy 3.8 y `spacy.blank`; pueden variar ligeramente con otras versiones.

### 3.3 Segmentación en frases

Separar frases parece fácil («cortar en cada punto») y no lo es: los puntos aparecen en abreviaturas (`Sr.`, `Dott.`, `p.m.`), siglas (`U.S.A.`) y decimales (`4.5`). `sent_tokenize` de NLTK usa un modelo entrenado (Punkt) que acierta mucho, pero no siempre.

In [ ]:
# datos-es
TEXTOS_FRASES = {
    "spa": "El Sr. García pagó 4.5 euros. Luego se fue, etc. Volvió a las 3 p.m. y dijo: ¿Qué tal? Bien. Vi a la U.S.A. con el Dr. Pérez.",
    "ita": "Il dott. Rossi ha pagato 4.5 euro. Poi è andato via, ecc. Tornò alle 3 e disse: Bene. Sig.ra Bianchi, buongiorno.",
}

In [ ]:
for idioma, texto in TEXTOS_FRASES.items():
    ingenua = [f.strip() for f in re.split(r"(?<=[.!?])\s+", texto)]
    punkt = sent_tokenize(texto, language=nombre_nltk[idioma])
    print(idioma, "regex ingenua :", len(ingenua), ingenua)
    print(idioma, "sent_tokenize :", len(punkt), punkt)

**¿Qué observas?** El texto en español tiene 5 frases y el italiano 4. La regex ingenua parte en cada abreviatura (9 y 5 «frases»). Punkt se equivoca menos (7 y 4), pero también falla: corta tras `p.m.` y `U.S.A.` en español, y en italiano corta tras `Dott.` y **une** dos frases tras `ecc.`; acierta el número, pero no los cortes (con la versión de NLTK usada aquí; puede variar entre versiones). **Regla práctica:** si la segmentación en frases importa (resúmenes, clase 11), revisa a mano una muestra de tus textos; si no importa, no la hagas.

---
## 4. Tokens y tipos

Un **token** es cada ocurrencia de una palabra en el texto; un **tipo** (*type*) es una palabra distinta del vocabulario. `el gato y el perro` tiene 5 tokens y 4 tipos. Con ellos se calculan la **frecuencia** de cada tipo, la **razón tipos/tokens** (TTR) y los ***hapax legomena***, los tipos que aparecen **una sola vez**. Veremos que unos pocos tipos concentran casi todos los tokens y que la mayoría de los tipos son hapax.

Calculamos las cifras para los dos libros (limpiados con `limpiar` y pasados a minúsculas) y para las reseñas. **Advertencia:** son textos de siglos y estilos distintos (el español del siglo XVII de Cervantes, el italiano del siglo XIX de Manzoni, y reseñas sintéticas de estilo actual), así que las diferencias no dicen nada sobre «el español» ni «el italiano».

In [ ]:
def palabras(texto):
    """Tokens de palabra en minúsculas (sin puntuación)."""
    return [t for t in tokenizar(texto.lower()) if re.match(r"\w", t)]

def resumen_tokens(tokens):
    conteo = Counter(tokens)
    hapax = sum(1 for v in conteo.values() if v == 1)
    return {"tokens": len(tokens), "tipos": len(conteo), "TTR": round(len(conteo) / len(tokens), 3),
            "hapax": hapax, "% hapax sobre tipos": round(100 * hapax / len(conteo), 1)}

corpus = {}
for idioma in ("spa", "ita"):
    corpus[f"libro {idioma}"] = palabras(limpiar(libros[idioma]))
    corpus[f"reseñas {idioma}"] = [p for t in resenas[idioma]["texto"] for p in palabras(limpiar(t))]

tabla = pd.DataFrame([resumen_tokens(tk) for tk in corpus.values()], index=list(corpus))
print(tabla.to_string())

**¿Qué observas?** El TTR **no es comparable** entre corpus de tamaños distintos: cuantos más tokens, más se repiten los tipos y más baja el cociente. Para comparar hay que fijar el mismo número de tokens (por ejemplo, los primeros 1 000). Lo hacemos a continuación y añadimos las 20 palabras más frecuentes.

In [ ]:
primeros = pd.DataFrame([resumen_tokens(tk[:1000]) for tk in corpus.values()], index=list(corpus))
print("Primeros 1 000 tokens de cada corpus:")
print(primeros.to_string())
print()
top = {nombre: Counter(tk).most_common(20) for nombre, tk in corpus.items()}
for nombre, lista in top.items():
    print(f"{nombre}:", ", ".join(f"{p} {n}" for p, n in lista))

### 4.1 La distribución rango-frecuencia

Ordenamos los tipos de mayor a menor frecuencia y dibujamos frecuencia frente a rango en escala doble logarítmica, junto a la curva de crecimiento del vocabulario (cuántos tipos hay tras leer *n* tokens).

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(11, 4))
for nombre in corpus:
    tokens = corpus[nombre]
    frecuencias = sorted(Counter(tokens).values(), reverse=True)
    ejes[0].loglog(range(1, len(frecuencias) + 1), frecuencias, label=nombre)
    crecimiento = [len(set(tokens[:n])) for n in range(100, len(tokens) + 1, 100)]
    ejes[1].plot(range(100, 100 * len(crecimiento) + 1, 100), crecimiento, label=nombre)
ejes[0].set_xlabel("rango (escala log)")
ejes[0].set_ylabel("frecuencia (escala log)")
ejes[0].set_title("Rango-frecuencia")
ejes[1].set_xlabel("tokens leídos")
ejes[1].set_ylabel("tipos distintos")
ejes[1].set_title("Crecimiento del vocabulario")
ejes[0].legend()
plt.tight_layout()
plt.show()

**¿Qué observas?** A la izquierda, casi una recta descendente en escala log-log: unos pocos tipos (`de`, `y`, `que`, `e`, `di`, `il`…) acumulan muchísimos tokens y una larga cola de tipos aparece una vez (la tendencia se conoce como ley de Zipf). A la derecha, la curva del vocabulario **sigue creciendo** sin estabilizarse: siempre aparecerán palabras nuevas, lo que en las clases 4 y 8 se traduce en el problema de las palabras fuera de vocabulario (OOV). Más de la mitad de los tipos son hapax en los cuatro corpus.

---
## 5. Medir la calidad de una limpieza

Las reseñas traen la columna `texto_limpio`: una versión de referencia (gold) escrita a mano. Con ella podemos **medir** una limpieza en vez de juzgarla a ojo. Tres métricas sencillas, comparando `limpiar(texto)` con `texto_limpio` **tras normalizar** (NFC, `casefold`, espacios), porque `limpiar` no restituye la capitalización:

1. **Coincidencia exacta**: porcentaje de reseñas cuyo resultado es idéntico al gold.
2. **F1 de tokens**: precisión y *recall* sobre el multiconjunto de palabras (media por reseña).
3. **Similitud** de `difflib.SequenceMatcher`: razón entre 0 y 1 a nivel de carácter.

In [ ]:
def normalizar_comparacion(texto):
    return re.sub(r"\s+", " ", unicodedata.normalize("NFC", texto).casefold()).strip()

def f1_tokens(pred, gold):
    """F1 entre dos listas de tokens (multiconjuntos)."""
    comunes = sum((Counter(pred) & Counter(gold)).values())
    if comunes == 0:
        return 0.0
    precision, recall = comunes / len(pred), comunes / len(gold)
    return 2 * precision * recall / (precision + recall)

def evaluar_limpieza(limpiador, textos, golds):
    pares = [(normalizar_comparacion(limpiador(t)), normalizar_comparacion(g)) for t, g in zip(textos, golds)]
    return {"exacta": round(np.mean([p == g for p, g in pares]), 3),
            "F1": round(np.mean([f1_tokens(palabras(p), palabras(g)) for p, g in pares]), 3),
            "similitud": round(np.mean([difflib.SequenceMatcher(None, p, g).ratio() for p, g in pares]), 3)}

def solo_espacios(texto):
    return arreglar_espacios(texto)

resultados = {}
for idioma in ("spa", "ita"):
    d = resenas[idioma]
    resultados[f"sin limpiar {idioma}"] = evaluar_limpieza(lambda t: t, d["texto"], d["texto_limpio"])
    resultados[f"solo espacios {idioma}"] = evaluar_limpieza(solo_espacios, d["texto"], d["texto_limpio"])
    resultados[f"limpiar {idioma}"] = evaluar_limpieza(limpiar, d["texto"], d["texto_limpio"])
    resultados[f"limpiar(conservar) {idioma}"] = evaluar_limpieza(lambda t: limpiar(t, conservar=True), d["texto"], d["texto_limpio"])
print(pd.DataFrame(resultados).T.to_string())

**¿Qué observas?** (1) `limpiar` mejora las tres métricas frente a no hacer nada. (2) La versión `conservar=True` **puntúa peor**: no es un error, es que el gold **borra** hashtags y emojis mientras nosotros los conservamos a propósito. Una métrica mide la distancia a *un* criterio; si el criterio no es el tuyo, la cifra engaña. (3) Aun así la exacta no llega al 100 %: queda lo que ninguna regla puede recuperar. Lo clasificamos.

In [ ]:
def sin_puntuacion(texto):
    return re.sub(r"[^\w\s]", " ", texto)

def causa_residuo(pred, gold):
    """Clasifica por qué una limpieza no coincide con el gold."""
    p, g = normalizar_comparacion(pred), normalizar_comparacion(gold)
    if p == g:
        return "coincide"
    if palabras(p) == palabras(g):
        return "solo puntuación"
    if palabras(quitar_acentos(p)) == palabras(quitar_acentos(g)):
        return "acentos perdidos"
    return "otras (erratas, abreviaturas)"

for idioma in ("spa", "ita"):
    d = resenas[idioma]
    causas = Counter(causa_residuo(limpiar(t), g) for t, g in zip(d["texto"], d["texto_limpio"]))
    print(idioma, {k: f"{100 * v / len(d):.1f}%" for k, v in causas.most_common()})

**¿Qué observas?** Una parte de las reseñas sigue sin coincidir por **puntuación** (el gold cambia `!!` por `.`: es una convención del gold, no un fallo), otra por **acentos perdidos** (`rompio` frente a `rompió`) y otra por **erratas y abreviaturas** (`esrellas`, `q`): la regex no puede saber que `esrellas` quiere decir `estrellas`. Recuperar acentos y corregir erratas exige un diccionario o un modelo (un corrector ortográfico), y aun así se equivocará a veces. **Lo importante es saber qué puede y qué no puede hacer la limpieza y medirlo.**

In [ ]:
# Ejemplos concretos de lo que la limpieza no recupera (español e italiano)
for idioma in ("spa", "ita"):
    d = resenas[idioma]
    vistos = 0
    for t, g in zip(d["texto"], d["texto_limpio"]):
        if causa_residuo(limpiar(t), g) != "coincide" and vistos < 3 and len(t) < 110:
            print(idioma, "|", limpiar(t))
            print("   gold |", g)
            vistos += 1

---
## 6. Otras dificultades del lenguaje (que la limpieza no arregla)

En la clase 1 vimos la ambigüedad léxica y sintáctica. El texto bruto añade tres fuentes más de dificultad: la **anáfora** (a quién se refiere un pronombre), el **contexto y la pragmática** (lo implícito y la ironía) y la **variación** (erratas, registros, emojis, mezcla de idiomas). La primera y la segunda solo las mostramos: son límites del enfoque clásico. La tercera sí la podemos **cuantificar** sobre las reseñas.

### 6.1 Ambigüedad referencial (anáfora)

In [ ]:
# datos-es
REFERENCIAL = [
    ("Ana le dijo a Marta que ella tenía razón", "Ana o Marta?"),
    ("El trofeo no cabe en la maleta porque es demasiado grande", "el trofeo"),
    ("El trofeo no cabe en la maleta porque es demasiado pequeña", "la maleta"),
    ("Il trofeo non entra nella valigia perché è troppo grande", "il trofeo"),
    ("Marco ha detto a Luca che lui aveva ragione", "Marco o Luca?"),
]

In [ ]:
for frase, referente in REFERENCIAL:
    print(f"{frase}  →  {referente}")

**¿Qué observas?** Cambiar una sola palabra (`grande` / `pequeña`) cambia a qué se refiere `es`; y en la primera y la última frase ni siquiera hay una respuesta única. Resolverlo exige **conocimiento del mundo**, no solo gramática: casi ningún método clásico lo hace bien, y es un límite que hay que conocer. Ninguna regex ni ninguna limpieza lo arregla.

### 6.2 Contexto y pragmática: la ironía en las reseñas

Lo que se dice no siempre es lo que se quiere decir: `¡Qué buen día para que se estropee el aire acondicionado!` es una queja; `Bravo, hai proprio scelto il momento giusto!` es un reproche. Lo comprobamos con las reseñas difíciles (`dificil == 1`) etiquetadas como **negativas** que contienen palabras de tono positivo: para una regla de palabras, parecerían positivas.

In [ ]:
# datos-es
TONO_POSITIVO = {"spa": {"maravilla", "fantástico", "genial", "perfecto", "gracias", "felicidades", "excelente"},
                 "ita": {"bravo", "complimenti", "grazie", "ottimo", "perfetto", "bellissima", "bellissimo", "eccellente"}}

In [ ]:
for idioma in ("spa", "ita"):
    d = resenas[idioma]
    dificiles = d[(d["dificil"] == 1) & (d["etiqueta"] == "negativo")]
    con_tono = dificiles[[bool(set(palabras(limpiar(t))) & TONO_POSITIVO[idioma]) for t in dificiles["texto"]]]
    print(f"{idioma}: {len(dificiles)} reseñas difíciles negativas; con palabras de tono positivo: {len(con_tono)}")
    for t in con_tono["texto"].head(3):
        print("   ·", limpiar(t))

**¿Qué observas?** Son quejas escritas con palabras elogiosas. Una limpieza impecable no las convierte en algo distinto: el problema no está en el texto bruto, está en la **pragmática**. Por eso el análisis de sentimiento tiene un techo (clase 6) y hay que analizar los errores a mano; y por eso, en la clase 12, se compara con un modelo neuronal que ve más contexto.

### 6.3 Variación: cuantificarla en las reseñas

La variación sí se puede **medir**: con contadores sencillos calculamos el porcentaje de reseñas que presentan cada fenómeno. Para los acentos perdidos usamos el gold (solo posible porque lo tenemos).

In [ ]:
# datos-es
CHAT = {"spa": {"q", "xq", "pq", "xk", "k", "tb", "tbn", "ke"}, "ita": {"ke", "xke", "nn", "sn", "cmq", "x", "qlc", "tvb"}}
TILDES = set("áéíóúàèìòùñÁÉÍÓÚÀÈÌÒÙ")

In [ ]:
def pct(condicion, textos):
    return round(100 * np.mean([bool(condicion(t)) for t in textos]), 1)

def variacion(idioma):
    d = resenas[idioma]
    textos, golds = d["texto"], d["texto_limpio"]
    n_tildes = lambda t: sum(c in TILDES for c in t)
    return {"alguna palabra en MAYÚSCULAS": pct(lambda t: senales(t, idioma)["mayúsculas"] > 0, textos),
            "acentos perdidos": round(100 * np.mean([n_tildes(t) < n_tildes(g) for t, g in zip(textos, golds)]), 1),
            "alargamientos": pct(RE_ALARGADA.search, textos),
            "emojis": pct(RE_EMOJI.search, textos),
            "URLs": pct(RE_URL.search, textos),
            "menciones o hashtags": pct(lambda t: re.search(r"[@#]\w", t), textos),
            "espacios dobles": pct(lambda t: "  " in t, textos),
            "abreviaturas de chat": pct(lambda t: set(re.findall(r"\w+", t.lower())) & CHAT[idioma], textos),
            "texto distinto del gold": round(100 * np.mean([t != g for t, g in zip(textos, golds)]), 1)}

tabla_variacion = pd.DataFrame({"spa": variacion("spa"), "ita": variacion("ita")})
print(tabla_variacion.to_string())
tabla_variacion.plot.barh(figsize=(9, 4.5))
plt.xlabel("% de reseñas")
plt.title("Variación en las reseñas brutas (300 por idioma)")
plt.tight_layout()
plt.show()

**¿Qué observas?** Casi la mitad de las reseñas difiere del gold por alguna causa, y cada fenómeno afecta a una fracción pequeña pero no despreciable. Cada variante es, para un programa que compara cadenas, una palabra distinta: `holaaa` y `hola`, `xq` y `porque`. La normalización (esta clase) y la lematización (clase 3) reducen parte del problema, pero no todo. Y son cifras de reseñas **sintéticas** con ruido inyectado: en textos reales las proporciones serán otras.

---
## 7. Resumen y siguiente paso

- **Unicode**: un carácter no es un byte; NFC y NFD dan cadenas distintas para lo que ves igual; normaliza a NFC al leer y usa `casefold()` para comparar.
- **Acentos**: quitarlos reduce variantes pero borra distinciones (`papá`/`papa`, `è`/`e`); hazlo solo con criterio y protege la `ñ`.
- **Limpieza**: pasos con nombre, cada uno probable por separado; **limpiar no es borrar**: extrae o sustituye por tokens lo que aporta información (emojis, mayúsculas, `!!!`).
- **Tokenización**: `split()` no basta; un patrón con `re.findall` gestiona apóstrofos, `¿ ¡`, cifras; NLTK y spaCy toman decisiones distintas.
- **Tokens y tipos**: pocos tipos concentran muchos tokens y más de la mitad de los tipos son hapax; el TTR depende del tamaño.
- **Evaluar** una limpieza contra un gold con exacta, F1 y similitud, y saber qué no puede recuperar (acentos, erratas).
- La **anáfora** y la **pragmática** no se resuelven limpiando: son límites que verás de nuevo en las clases 6 y 12.

**Siguiente:** en la clase 3 quitarás *stopwords* con criterio y compararás *stemming* y lematización. Antes, haz el notebook de práctica de esta clase.